# Session 5 · Break it, fix it

1. Run the **10 evaluation questions** and get a results table.
2. Find out **why** the red rows fail.
3. Fix them, run the questions again, and compare.

The fixes go in the cells marked **Fix 1/2/3**. Each one replaces one small function of the project.

In [ ]:
# Run this first. In Colab it downloads the workshop code; on a laptop it finds the project folder.
import os, subprocess, sys

REPO_URL = "https://github.com/1Baleid/tuwaiq-agent-workshop.git"   # the student repo
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/workshop"):
        subprocess.run(["git", "clone", "--quiet", REPO_URL, "/content/workshop"], check=True)
    os.chdir("/content/workshop")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements-colab.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # the code lives one folder up
sys.path.insert(0, os.getcwd())
print("Project folder:", os.getcwd())

This notebook uses your code from Sessions 1–4. If some of it is unfinished, take the working
versions first: remove the `#` in the next cell and run it.

In [ ]:
# !git checkout origin/solutions -- rag.py answer.py tools.py agent.py llm.py

## 1. Which LLM?

In [ ]:
import config

# config.LLM_PROVIDER = "local"   # Plan B, only if the instructor says so (see 00_setup)
config.VERBOSE = False            # True prints every step the agent takes
print("Provider:", config.LLM_PROVIDER, "| model:", config.PROVIDERS[config.LLM_PROVIDER]["model"])

## 2. Run the evaluation

Each question takes a few seconds. A row passes when the answer contains the expected words
**and** cites the right source (or used the right tool).

In [ ]:
import eval_runner

rows_before = eval_runner.run_eval()
eval_runner.results_table(rows_before)

## 3. Find out why

For each red row, run `explain("q…")` below and read what the agent did:

* Did it search the documents? What came back?
* Did it call a tool? With which arguments? What did the tool answer?

Then decide: is the problem in the **documents**, a **tool**, or the **instructions**?
Useful too: `rag.load_documents()` shows which files were read.

In [ ]:
import agent

def explain(question_id):
    """Ask one evaluation question again and print every step the agent takes."""
    question = next(q for q in eval_runner.load_questions() if q["id"] == question_id)
    config.VERBOSE = True
    result = agent.run_agent(question["question"])
    config.VERBOSE = False
    print("\nExpected:", question["expected_answer"])

explain("q8")

## 4. Fix 1: a document the loader does not read (q8)

The answer is in `data/sample/docs/library_rules.html`, but `rag.read_file()` only reads `.txt`, `.md`
and `.pdf` files, so the page never reaches the index. Teach the loader to read `.html` pages,
then rebuild the index.

In [ ]:
# SOLUTION START: TODO (Session 5): write read_file(path) that also reads .html pages (drop the tags), set rag.read_file = read_file, and rebuild with rag.get_store(rebuild=True)
import html
import re

import rag


def read_file(path):
    """Like rag.read_file, plus .html pages: keep the words, drop the tags."""
    suffix = path.suffix.lower()
    if suffix in (".txt", ".md"):
        return path.read_text(encoding="utf-8")
    if suffix == ".pdf":
        from pypdf import PdfReader
        return "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)
    if suffix in (".html", ".htm"):
        page = re.sub(r"(?is)<(script|style).*?</\1>", " ", path.read_text(encoding="utf-8"))
        text = re.sub(r"<[^>]+>", " ", page)                  # remove the tags
        return re.sub(r"\s+", " ", html.unescape(text)).strip()  # &amp; -> &, tidy the spaces
    return None


rag.read_file = read_file
rag.get_store(rebuild=True)
# SOLUTION END

## 5. Fix 2: deadline names must match exactly (q9)

The student asks about the "last day to add or drop a course", and the agent calls
`get_deadline` with a name like "add/drop deadline". But `tools.find_deadline()` only accepts the exact
name in `deadlines.csv` ("Add/drop period ends"). Make it find the closest name instead.

In [ ]:
# SOLUTION START: TODO (Session 5): write find_deadline(name, deadlines) that returns the deadline sharing the most words with name (or None), then set tools.find_deadline = find_deadline
import re

import tools


def words(text):
    return set(re.findall(r"[a-z]+", text.lower()))


def find_deadline(name, deadlines):
    """The deadline that shares the most words with `name`, e.g. 'add or drop' finds 'Add/drop period ends'."""
    best = max(deadlines, key=lambda known: len(words(name) & words(known)))
    return best if words(name) & words(best) else None


tools.find_deadline = find_deadline
print(tools.get_deadline("add/drop deadline"))
# SOLUTION END

## 6. Fix 3: a date typed the wrong way (q10)

`get_deadline` now finds "Course withdrawal deadline", but says its date is not valid. Look at that row
in `data/sample/deadlines.csv`: someone typed `17/12/2026` instead of `2026-12-17`. Real data is like
this. Make `tools.parse_date()` accept both ways of writing a date.

In [ ]:
# SOLUTION START: TODO (Session 5): write parse_date(text) that accepts 2026-12-17 and 17/12/2026, then set tools.parse_date = parse_date
from datetime import datetime

import tools


def parse_date(text):
    """Read a date written as 2026-12-17, 17/12/2026 or 17-12-2026."""
    for pattern in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return datetime.strptime(text.strip(), pattern).date()
        except ValueError:
            continue
    raise ValueError(f"unknown date format: {text}")


tools.parse_date = parse_date
print(tools.get_deadline("Course withdrawal deadline"))
# SOLUTION END

## 7. Run again and compare

In [ ]:
rows_after = eval_runner.run_eval()
print(f"Before: {sum(r['passed'] for r in rows_before)}/{len(rows_before)} passed. "
      f"After: {sum(r['passed'] for r in rows_after)}/{len(rows_after)} passed.")
eval_runner.results_table(rows_after)

## 8. Save the results (for your demo)

In [ ]:
eval_runner.save_results(rows_before, "before")
eval_runner.save_results(rows_after, "after")